# MetaPhlAn3 tutorial
MetaPhlAn relies on unique clade-specific marker genes that were identified from ~17,000 reference genomes (further details)

*   ~13,500 bacteria and archaea
*   ~3,500 viruses
*   ~110 eukaryotes

Illustration of clades [here](https://drive.google.com/file/d/1Ln_-JTfAG6fNuSl9RpLKjUQeynL229OK/view?usp=drive_link)

# Workflow:

**Input**:


*   QC'ed reads
*   Clade-specific marker gene database


**Output**:


*   Marker genes mapping file [.sam]
*   Microbial profile table [.txt]





In [ ]:
from google.colab import drive
import os,glob
drive.mount('/content/drive')
py_env='/content/drive/MyDrive/workshop_June2023/reference_free_metagenomic_analysis/python_env'
root_dir = "/content/drive/MyDrive/workshop_June2023"
metaphlan_source_dir = os.path.join(root_dir,"reference_based_metagenomic_analysis","MetaPhlAn-3.1.0")

Mounted at /content/drive


# Setup MetaPhlAn

In [ ]:
# install bowtie2
! sudo apt install bowtie2
# install metaphlan
%cd {metaphlan_source_dir}
! pwd
! pip install .

In [ ]:
# check if the installation is successful
! which bowtie2
! which metaphlan

/usr/bin/bowtie2
/usr/local/bin/metaphlan


# Input data:

In [ ]:
# mgx reads folder
mgx_reads_dir = os.path.join(root_dir,"mgx_reads")
! ls -lh {mgx_reads_dir}
sample_id="PSMB4MBK"
metaphlan_profile_dir =  os.path.join(root_dir,"reference_based_metagenomic_analysis","profile")
! rm -rf stdin_map.bowtie2out.txt
! mkdir -p {metaphlan_profile_dir}
! zcat {mgx_reads_dir}/{sample_id}_R1.fastq.gz | head

total 780M
-rw------- 1 root root  15K May  3 04:55 PSMB4MBK.log
-rw------- 1 root root 389M May  3 04:55 PSMB4MBK_R1.fastq.gz
-rw------- 1 root root 391M May  3 04:55 PSMB4MBK_R2.fastq.gz
-rw------- 1 root root 482K Jun 12 14:25 PSMB4MBK_subset_R1.fastq.gz
-rw------- 1 root root 478K Jun 12 14:27 PSMB4MBK_subset_R2.fastq.gz
@CAVL1ANXX170419:1:1101:10000:97843/1
CGAAGAGCCTTCGCGTGATCGGCATCAGGAACTTGGGATTGGCGGGATCG
+
<BBBBFFFFFFFFFFFFFFFFFFBBFFFFFFFFFFFBFFFFFFFFFFFFF
@CAVL1ANXX170419:1:1101:10002:84329/1
CCAGTGCACAGAGCAGATAATAGGAAGAATCATCTGCATAGGCCAGACGGTTTGCACGGTCATTGATGAGACCGTATTTGGCAGAAAAGCTGTCATAGAGG
+
/<<BBFFFFFFFFFFFFFFFFFFFFFFF<FFFFFFFFFFFFFFFFFFFF<7FBFFFFFFF<BFBBFFFFFFF<FFFFFFFFFFFFFFFFFFFFFFFFFF/B
@CAVL1ANXX170419:1:1101:10004:100339/1
CCACAGAAAAAGCACTTTCCTCATAAAAGCTGTTGTCCCAGTCTATTCCCTTTTCTTCCAGAACATCCAGAAATCCACGGAAACGTTCTTCCCCGGTTGCA


## Count the number of lines and the number of reads in the fastq file

In [ ]:
! zcat {mgx_reads_dir}/{sample_id}_R1.fastq.gz | wc -l

34497952


In [ ]:
34497952/4

8624488.0

## Reminder: fastq format

Four lines per record [per read]


1.   Read identifier [starts with @, ends with /1 or /2]
2.   Nucleotide sequence
3.   Place holder / separater [+]
4.   Phred score [quality score for each nucleotide position]

Phred score table [here](https://drive.google.com/file/d/1poJ6joQKbhJO9btjvV5xEvRr2_0uGAxs/view?usp=drive_link)

## The Phred score encoding system:

image [here](https://drive.google.com/file/d/1WNVk2h8AFKKcXyxSA2h0dyEAcRlVkKzL/view?usp=drive_link)

# Running MetaPhlAn

In [ ]:
# metaphlan database
metaphlan_db_dir = os.path.join(root_dir,"reference_based_metagenomic_analysis","Metaphlan3_DB")
! ls -lh {metaphlan_db_dir}

total 3.3G
-rw------- 1 root root   26 Jun  2 13:03 mpa_latest
-rw------- 1 root root 727M Jun 12 12:03 mpa_v31_CHOCOPhlAn_201901.1.bt2
-rw------- 1 root root 336M Jun 12 12:03 mpa_v31_CHOCOPhlAn_201901.2.bt2
-rw------- 1 root root  12M Jun 12 12:58 mpa_v31_CHOCOPhlAn_201901.3.bt2
-rw------- 1 root root 336M Jun 12 11:49 mpa_v31_CHOCOPhlAn_201901.4.bt2
-rw------- 1 root root 379M Jun  2 13:03 mpa_v31_CHOCOPhlAn_201901.fna.bz2
-rw------- 1 root root   64 Jun  2 13:02 mpa_v31_CHOCOPhlAn_201901.md5
-rw------- 1 root root  29M Jun  2 13:02 mpa_v31_CHOCOPhlAn_201901.pkl
-rw------- 1 root root 727M Jun 12 12:58 mpa_v31_CHOCOPhlAn_201901.rev.1.bt2
-rw------- 1 root root 336M Jun 12 12:58 mpa_v31_CHOCOPhlAn_201901.rev.2.bt2
-rw------- 1 root root 408M Jun  2 13:03 mpa_v31_CHOCOPhlAn_201901.tar
-rw------- 1 root root   50 Jun  2 13:03 README.txt


## Command format:

In [ ]:
! metaphlan --help

Relavent parameters
*   rel_ab_w_read_stats: profiling relative abundances and estimating the number of reads coming from each clade.
*   nproc: The number of CPUs to use for parallelizing the mapping step [default 4]




In [ ]:
# check input fastq files
! ls -lh {mgx_reads_dir}/{sample_id}_R*.fastq.gz
# create the output folder
! mkdir -p {metaphlan_profile_dir}

-rw------- 1 root root 389M May  3 04:55 /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_R1.fastq.gz
-rw------- 1 root root 391M May  3 04:55 /content/drive/MyDrive/workshop_June2023/mgx_reads/PSMB4MBK_R2.fastq.gz


In [ ]:

! zcat {mgx_reads_dir}/{sample_id}_R*.fastq.gz | metaphlan --bowtie2db {metaphlan_db_dir} -x mpa_v31_CHOCOPhlAn_201901 -t rel_ab_w_read_stats --nproc 8 --input_type fastq > {metaphlan_profile_dir}/{sample_id}.profile.txt;


## Check the output:

In [ ]:
! ls -lh {metaphlan_profile_dir}/{sample_id}.profile.txt;
! head {metaphlan_profile_dir}/{sample_id}.profile.txt

-rw------- 1 root root 23K Jun 12 13:53 /content/drive/MyDrive/workshop_June2023/reference_based_metagenomic_analysis/profile/PSMB4MBK.profile.txt
#mpa_v31_CHOCOPhlAn_201901
#/usr/local/bin/metaphlan --bowtie2db /content/drive/MyDrive/workshop_June2023/reference_based_metagenomic_analysis/Metaphlan3_DB -x mpa_v31_CHOCOPhlAn_201901 -t rel_ab_w_read_stats --nproc 8 --input_type fastq
#SampleID	Metaphlan_Analysis
#estimated_reads_mapped_to_known_clades:7731682
#clade_name	clade_taxid	relative_abundance	coverage	estimated_number_of_reads_from_the_clade
k__Bacteria	2	100.0	1.64615	5896762
k__Bacteria|p__Firmicutes	2|1239	66.82653	1.10006	3446758
k__Bacteria|p__Bacteroidetes	2|976	31.36166	0.51626	2382083
k__Bacteria|p__Actinobacteria	2|201174	1.77148	0.02916	66557
k__Bacteria|p__Proteobacteria	2|1224	0.04033	0.00066	1364


In [ ]:
# An example of bacterial lineage
! grep s__ -m1 {metaphlan_profile_dir}/{sample_id}.profile.txt #| cut -f1 | sed 's/|/\n/g'

k__Bacteria|p__Firmicutes|c__Clostridia|o__Clostridiales|f__Ruminococcaceae|g__Faecalibacterium|s__Faecalibacterium_prausnitzii	2|1239|186801|186802|541000|216851|853	25.44958	0.41894	1254052


## Merging tables from multiple samples

In [ ]:
! ls -lh {metaphlan_profile_dir}/*.profile3.txt

-rw------- 1 root root 8.8K Jun  4 21:11 /content/drive/MyDrive/workshop_June2023/reference_based_metagenomic_analysis/profile/CSM5FZ3T_P.profile3.txt
-rw------- 1 root root  27K Jun  4 21:11 /content/drive/MyDrive/workshop_June2023/reference_based_metagenomic_analysis/profile/CSM5FZ3V_P.profile3.txt


In [ ]:
! merge_metaphlan_tables.py {metaphlan_profile_dir}/*profile3.txt > {metaphlan_profile_dir}/merged_abundance_table.txt

In [ ]:
# check the output
! head {metaphlan_profile_dir}/merged_abundance_table.txt


#mpa_v30_CHOCOPhlAn_201901
clade_name	NCBI_tax_id	CSM5FZ3V_P.profile3	CSM5FZ3T_P.profile3
k__Bacteria	2	100.0	100.0
k__Bacteria|p__Bacteroidetes	2|976	92.7779	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia	2|976|200643	92.7779	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales	2|976|200643|171549	92.7779	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae	2|976|200643|171549|815	92.6905	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides	2|976|200643|171549|815|816	92.6905	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_caccae	2|976|200643|171549|815|816|47678	2.70692	0
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_dorei	2|976|200643|171549|815|816|357276	0.84337	0.28864


In [ ]:
# check species level abundance
! grep "s__" {metaphlan_profile_dir}/merged_abundance_table.txt | head

k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_caccae	2|976|200643|171549|815|816|47678	2.70692	0
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_dorei	2|976|200643|171549|815|816|357276	0.84337	0.28864
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_fragilis	2|976|200643|171549|815|816|817	9.18445	11.08387
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_ovatus	2|976|200643|171549|815|816|28116	0.0776	0
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_salyersiae	2|976|200643|171549|815|816|291644	0.01008	0
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_thetaiotaomicron	2|976|200643|171549|815|816|818	11.28384	13.55901
k__Bact

In [ ]:
# check phylym level abundance
! grep "p__" {metaphlan_profile_dir}/merged_abundance_table.txt | grep -v "c__"


k__Bacteria|p__Bacteroidetes	2|976	92.7779	95.6901
k__Bacteria|p__Firmicutes	2|1239	6.4211	3.8043
k__Bacteria|p__Proteobacteria	2|1224	0.03646	0.5056
k__Bacteria|p__Verrucomicrobia	2|74201	0.76454	0


In [ ]:
# check order Bacteroidales at family level abundance
! grep "o__Bacteroidales" {metaphlan_profile_dir}/merged_abundance_table.txt | grep -v "g__"

k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales	2|976|200643|171549	92.7779	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae	2|976|200643|171549|815	92.6905	95.6901
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Rikenellaceae	2|976|200643|171549|171550	0.07243	0
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Tannerellaceae	2|976|200643|171549|2005525	0.01497	0


## Task: how many species were identified?

Hint:


*   **grep** species-level records
*   use **wc** command to count the number of records



In [ ]:
# solution
! grep "s__" {metaphlan_profile_dir}/merged_abundance_table.txt | wc -l

48
